# Conversion Unimarc vers matrice excel

Import :
- dans l'état du code il faut le fichier SRUextraction.py pour avoir les fonctions d'interrogation du SRU BnF (uniquement utilisé a priori pour récupérer les NNA à partir des ISNI, dont pas nécessaire si finalement le mode de récupération des NNA est différent)
- j'ai ajouté des lignes d'installation pour que le notebook puisse tourner indépendamment des librairies déjà installées. 

In [118]:
import SRUextractionMODIFIE as sru
try : 
    import pandas
except ModuleNotFoundError:
    %pip install pandas
    import pandas
import re
from collections import Counter

try:
    from pymarc import MARCReader, Field
except ModuleNotFoundError:
    %pip install pymarc
    from pymarc import MARCReader, Field

try:
    from tqdm import tqdm
except ModuleNotFoundError:
    %pip install tqdm
    from tqdm import tqdm

try:
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
except ModuleNotFoundError:
    %pip install openpyxl
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE


### Référentiels
Ici, on commence par prendre les référentiels intermarc NG. 
Dans le cas des codes fonctions et des catégories techniques, les informations sont exprimées de façon différentes en Unimarc et en Intermarc NG : les dictionnaires ont pour but de permettre leur conversion.
Attention, ces dictionnaires ne sont pas exhaustifs : les seuls cas de figures pris en compte ici sont ceux explicitement discutés avec la BML. S'il y a utilisation d'autres codes fonctions ou d'autres informations en 200$b, la conversion ne sera pas satisfaisante.

In [119]:
CODES_PAYS = ["##", "ad", "ae", "af", "ag", "ai", "al", "am", "ao", "aq", "ar", "as", "at", "au", "aw", "ax", "az", "ba", "bb", "bd", "be", "bf", "bf", "bf", "bg", "bh", "bi", "bj", "bl", "bm", "bn", "bo", "bq", "bq", "bq", "br", "bs", "bt", "bv", "bw", "by", "by", "bz", "ca", "cc", "cc", "cd", "cd", "cf", "cf", "cf", "cg", "cg", "ch", "ch", "ci", "ck", "cl", "cm", "cn", "cn", "co", "cr", "cu", "cv", "cw", "cx", "cy", "cz", "cz", "de", "de", "dj", "dk", "dm", "do", "do", "dz", "ec", "ee", "eg", "eh", "er", "es", "et", "eu", "fi", "fj", "fk", "fk", "fm", "fm", "fo", "fr", "ga", "gb", "gb", "gd", "ge", "gf", "gg", "gh", "gi", "gl", "gm", "gn", "gp", "gq", "gr", "gs", "gt", "gu", "gw", "gy", "hk", "hm", "hn", "hr", "ht", "hu", "id", "ie", "il", "im", "im", "in", "io", "io", "io", "iq", "ir", "is", "it", "je", "jm", "jo", "jp", "ke", "kg", "kh", "ki", "km", "kn", "kn", "kp", "kp", "kr", "kr", "kw", "ky", "ky", "ky", "kz", "la", "lb", "lc", "li", "lk", "lr", "ls", "lt", "lu", "lv", "ly", "ly", "ma", "mc", "md", "md", "me", "mf", "mg", "mh", "mk", "ml", "mm", "mm", "mn", "mo", "mp", "mq", "mr", "ms", "mt", "mu", "mv", "mw", "mx", "my", "mz", "na", "nc", "ne", "nf", "ng", "ni", "nl", "no", "np", "nr", "nu", "nz", "om", "pa", "pe", "pf", "pg", "ph", "pk", "pl", "pm", "pn", "pr", "ps", "ps", "pt", "pw", "pw", "py", "qa", "re", "ro", "rs", "ru", "ru", "rw", "sa", "sb", "sc", "sd", "se", "sg", "sh", "sh", "sh", "si", "sj", "sk", "sl", "sm", "sn", "so", "sr", "ss", "ss", "st", "sv", "sv", "sx", "sx", "sy", "sz", "tc", "td", "tf", "tg", "th", "tj", "tk", "tl", "tm", "tn", "to", "tr", "tt", "tv", "tw", "tw", "tz", "ua", "ug", "um", "us", "uy", "uz", "va", "va", "vc", "ve", "vg", "vg", "vi", "vi", "vn", "vu", "wf", "ws", "ws", "xk", "ye", "ye", "yt", "za", "zm", "zw", "zz"]


In [120]:
CODES_LANGUE = ["###", "aar", "abk", "ace", "ace", "ach", "ach", "ach", "ada", "ada", "ada", "ada", "ady", "afa", "afa", "afh", "afr", "ain", "aka", "aka", "aka", "akk", "akk", "akk", "alb", "alb", "ale", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alt", "amh", "ang", "ang", "ang", "ang", "anp", "apa", "apa", "ara", "arc", "arc", "arc", "arg", "arm", "arn", "arn", "arn", "arp", "art", "art", "art", "arw", "asm", "ast", "ast", "ast", "ast", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "aus", "aus", "aus", "aus", "aus", "aus", "aus", "ava", "ave", "ave", "ave", "ave", "ave", "ave", "awa", "aym", "aze", "aze", "bad", "bad", "bai", "bai", "bak", "bak", "bal", "bam", "bam", "ban", "baq", "baq", "baq", "bas", "bas", "bas", "bas", "bas", "bas", "bat", "bat", "bej", "bej", "bej", "bel", "bem", "ben", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "bho", "bih", "bih", "bik", "bin", "bin", "bis", "bis", "bis", "bla", "bla", "bla", "bnt", "bnt", "bnt", "bnt", "bos", "bra", "bra", "bre", "bre", "bre", "bre", "bre", "bre", "btk", "btk", "bua", "bug", "bul", "bur", "byn", "byn", "byn", "cad", "cad", "cad", "cai", "cai", "car", "car", "car", "car", "cat", "cat", "cau", "cau", "cau", "cau", "cau", "cau", "ceb", "cel", "cel", "cel", "cel", "cel", "cel", "cha", "chb", "che", "chg", "chg", "chg", "chi", "chk", "chk", "chm", "chn", "chn", "chn", "chn", "cho", "chp", "chr", "chu", "chu", "chu", "chu", "chu", "chu", "chv", "chy", "cmc", "cmc", "cop", "cor", "cos", "cpe", "cpe", "cpe", "cpe", "cpe", "cpe", "cpe", "cpe", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpp", "cpp", "cpp", "cpp", "cpp", "cre", "cre", "cre", "crh", "crh", "crp", "crp", "crp", "crp", "crp", "crp", "crp", "csb", "cus", "cus", "cus", "cus", "cze", "dak", "dak", "dak", "dak", "dak", "dak", "dak", "dan", "dar", "day", "day", "del", "den", "den", "dgr", "dgr", "dgr", "din", "din", "div", "doi", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dsb", "dua", "dum", "dum", "dum", "dut", "dut", "dyu", "dyu", "dyu", "dzo", "dzo", "efi", "efi", "egy", "egy", "egy", "egy", "eka", "elx", "elx", "elx", "elx", "elx", "elx", "elx", "eng", "enm", "enm", "enm", "epo", "est", "ewe", "ewe", "ewe", "ewo", "fan", "fan", "fao", "fat", "fij", "fil", "fil", "fin", "fin", "fin", "fiu", "fiu", "fiu", "fiu", "fiu", "fon", "frdbnf", "frdbnf", "fre", "frm", "frm", "frm", "fro", "fro", "fro", "fro", "frr", "frs", "fry", "ful", "ful", "ful", "ful", "ful", "fur", "gaa", "gay", "gba", "gedbnf", "gedbnf", "gem", "gem", "gem", "gem", "gem", "gem", "geo", "ger", "gez", "gez", "gez", "gez", "gez", "gez", "gil", "gil", "gla", "gla", "gla", "gla", "gle", "gle", "gle", "gle", "glg", "glv", "glv", "glv", "gmh", "gmh", "goh", "goh", "gon", "gon", "gor", "got", "grb", "grc", "grc", "grc", "grc", "grc", "grc", "grc", "grc", "grdbnf", "grdbnf", "gre", "gre", "grm", "grn", "grp", "gsw", "gsw", "gsw", "gsw", "gsw", "gsw", "guj", "guj", "guj", "gwi", "hai", "hat", "hat", "hat", "hau", "haw", "heb", "heb", "her", "hil", "him", "him", "him", "hin", "hit", "hmn", "hmo", "hrv", "hsb", "hsb", "hun", "hun", "hup", "iba", "ibo", "ibo", "ice", "ido", "iii", "ijo", "ijo", "ijo", "ijo", "iku", "ile", "ilo", "ina", "ina", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "ind", "ind", "ine", "ine", "ine", "ine", "ine", "ine", "ine", "inh", "ipk", "ira", "ira", "ira", "ira", "ira", "ira", "ira", "ira", "iro", "iro", "iro", "iro", "iro", "iro", "iro", "iro", "ita", "itdbnf", "itdbnf", "itdbnf", "itdbnf", "jav", "jbo", "jpn", "jpr", "jrb", "kaa", "kab", "kab", "kac", "kac", "kal", "kal", "kam", "kan", "kan", "kan", "kan", "kar", "kar", "kas", "kas", "kas", "kas", "kau", "kaw", "kaw", "kaw", "kaz", "kbd", "kha", "khi", "khi", "khi", "khi", "khi", "khm", "khm", "kho", "kho", "kho", "kik", "kin", "kin", "kir", "kmb", "kmb", "kok", "kom", "kon", "kon", "kor", "kos", "kpe", "kpe", "kpe", "krc", "krl", "kro", "kro", "kro", "kro", "kro", "kro", "kro", "kru", "kua", "kua", "kum", "kur", "kut", "kut", "lad", "lad", "lad", "lah", "lah", "lah", "lam", "lao", "lao", "lat", "lav", "lav", "lav", "lez", "lim", "lin", "lit", "lol", "lol", "lol", "loz", "lsf", "ltz", "lua", "lub", "lub", "lub", "lug", "lug", "lui", "lun", "luo", "luo", "luo", "lus", "mac", "mad", "mag", "mag", "mag", "mag", "mah", "mai", "mak", "mal", "man", "man", "man", "man", "mao", "map", "map", "map", "map", "map", "map", "mar", "mar", "mas", "mas", "may", "mdf", "mdr", "men", "mga", "mga", "mga", "mga", "mic", "mic", "min", "min", "mis", "mis", "mkh", "mkh", "mlg", "mlt", "mnc", "mni", "mni", "mno", "mno", "moh", "mon", "mos", "mos", "mun", "mun", "mun", "mun", "mun", "mun", "mun", "mun", "mun", "mus", "mus", "mwl", "mwr", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myv", "nah", "nah", "nah", "nah", "nah", "nai", "nai", "nai", "nap", "nau", "nav", "nbl", "nde", "ndo", "ndo", "nds", "nds", "nds", "nds", "nep", "new", "new", "new", "nia", "nic", "nic", "nic", "nic", "nic", "niu", "niu", "nno", "nno", "nno", "nno", "nob", "nob", "nog", "nog", "non", "non", "non", "non", "non", "non", "non", "non", "nor", "nqo", "nso", "nso", "nso", "nub", "nub", "nub", "nub", "nub", "nub", "nwc", "nya", "nya", "nya", "nya", "nya", "nym", "nyn", "nyn", "nyo", "nzi", "nzi", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oji", "oji", "oji", "oji", "ori", "orm", "orm", "osa", "oss", "ota", "ota", "ota", "ota", "oto", "oto", "oto", "oto", "oto", "oto", "oto", "oto", "paa", "paa", "paa", "pag", "pal", "pal", "pal", "pal", "pal", "pam", "pan", "pan", "pan", "pap", "pau", "pau", "peo", "peo", "per", "per", "per", "per", "per", "phi", "phi", "phn", "pli", "pol", "pon", "por", "pra", "pra", "pro", "pro", "pro", "pus", "pus", "pus", "pus", "que", "que", "que", "raj", "rap", "rar", "rar", "roa", "roa", "roa", "roa", "roa", "roh", "roh", "roh", "rom", "rom", "rom", "rom", "rom", "rum", "rum", "run", "run", "run", "rup", "rup", "rus", "sad", "sag", "sah", "sah", "sah", "sai", "sai", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sam", "san", "san", "sas", "sat", "scn", "sco", "sco", "sel", "sel", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sga", "sga", "sga", "sgn", "shn", "shn", "sid", "sin", "sin", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sit", "sit", "sit", "sit", "sit", "sit", "sit", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "slo", "slv", "sma", "sme", "smi", "smi", "smi", "smi", "smj", "smn", "smo", "sms", "sna", "sna", "snd", "snk", "sog", "som", "son", "son", "sot", "sot", "sot", "sot", "spa", "spa", "spdbnf", "spdbnf", "spdbnf", "srd", "srn", "srp", "srr", "srr", "ssa", "ssa", "ssa", "ssa", "ssa", "ssw", "ssw", "ssw", "suk", "sun", "sun", "sus", "sus", "sus", "sux", "swa", "swa", "swe", "swe", "syc", "syr", "syr", "tah", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tam", "tam", "tat", "tel", "tel", "tel", "tem", "tem", "tem", "ter", "ter", "tet", "tgk", "tgl", "tgl", "tha", "tha", "tib", "tig", "tir", "tir", "tiv", "tiv", "tiv", "tkl", "tlh", "tli", "tmh", "tmh", "tog", "tog", "ton", "tpi", "tsi", "tsi", "tsn", "tsn", "tso", "tuk", "tuk", "tum", "tup", "tup", "tur", "tut", "tut", "tut", "tut", "tut", "tut", "tvl", "twi", "twi", "tyv", "udm", "udm", "udm", "uga", "uga", "uig", "ukr", "umb", "umb", "umb", "urd", "urd", "uzb", "uzb", "vai", "vai", "vai", "ven", "vie", "vie", "vol", "vot", "wak", "wak", "wak", "wak", "wak", "wak", "wak", "wak", "wal", "wal", "wal", "war", "was", "wel", "wel", "wen", "wen", "wen", "wln", "wol", "wol", "wol", "xal", "xal", "xho", "xho", "xho", "yao", "yap", "yap", "yid", "yid", "yor", "ypk", "ypk", "zap", "zbl", "zbl", "zen", "zha", "zha", "znd", "znd", "zul", "zun", "zza", "zza", "zza", "zza", "zza", "zza"]

In [121]:
CODES_FONCTION = {"570": "9990",
                  "070" : "0990",
                  "440" : "0440",
                  "260": "4210",
                  "390": "4010",
                  "320": "4040",
                  "340" : "9990",
                  "350": "0410",
                  "360" : "0410",
                  "530" : "0410",
                  "760" : "0410",
                  "510" : "0480",
                  "600" : "0530",
                  "610" : "3060",
                  "620" : "3060",
                  "740" : "2310",
                  "750": "2310",
                 "" : "9990"
                 }
                 
#Trouver qq chose pour code fonction Unimarc 560

Référentiel catégorie technique : ici, on utilise les valeurs renseignées en 200\\$b (saisie manuelle) pour alimenter le référentiel "catégorie technique" de la BnF de façon minimale. Si la 200\\$b n'a pas été correctement saisie, cela risque de créer des anomalies

In [122]:
CATEGORIE_TECHNIQUE = {
    "Photographie": "photographie",
    "Carte postale" : "procédé photomécanique",
    "Affiche" : "procédé photomécanique",
    "Estampe" : "estampe",
    "Manuscrit" : "dessin"
}


### Création d'une classe records pour les notices unimarcs

#### Fonctions utilisées pour créer la classe records

In [123]:
def field_to_string(field):
    
    res = ""
    subfields = field.subfields
    for subfield in subfields:
        res += f"${subfield.code} {subfield.value} "
    
    return res

In [124]:
def note_exe2string(zone_note, string):
    for note in zone_note:
        for subfield in note.get_subfields("a"):
            string += subfield if subfield.endswith(".") else subfield + "."
           
    return string
    

#### Passer l'indexation Rameau en chaîne de caractères
Le but des deux fonctions suivantes est de permettre de passer d'une indexation Rameau en unimarc à des chaînes de caractères, qu'on chargera en indexation non contrôlée en Intermarc NG

In [125]:
def liste_subfields(field):
    liste_subfields = []
    for e in field.split("$"):
        if e:
            liste_subfields.append((e[0], e[1:].strip()))
    return liste_subfields

def marcfield2label(field):
    if field:
        repetitions = {"a": " -- ",
                    "x": " -- ",
                    "y": " -- ",
                    "l": " / ",
                    "d": " ; "}
        for char in repetitions:
            if re.search(f"\${char}([^$]+) \${char}", field) is not None:
                field = re.sub(f"\${char}([^$]+) \${char}", f"${char}\\1 {repetitions[char]}", field)
            if re.search(f"\${char}([^$]+) \${char}", field) is not None:
                field = re.sub(f"\${char}([^$]+) \${char}", f"${char}\\1 {repetitions[char]}", field)
        rules_multi = {"d k j": "(d-k-j)",
                       "f e": "f e"}  # date précise
        rules = {"a": "a",
                "b": ". b",
                "c": " c",
                "x": " -- x",
                "y": " -- y",
                "5": " -- 5",
                "i": " (i)",
                "k": "-k",
                "j": "-j",
                "d": " d",
                "h": " h",
                "t": " t",
                "l": " (l)",
                "m": ". m",
                "n": ". n",
                "z": " -- z",
                "f": " (f)",
                "v": " (v)"
        }
        for rule in rules_multi:
            chars = rules_multi[rule].split(" ")
            if len(chars) == 2:
                if re.search(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+)", field) is not None:
                    field = re.sub(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+)", r"(\1 ; \2)", field)
            if len(chars) == 3:
                if re.search(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+) \${chars[2]}([^$]+)", field) is not None:
                    field = re.sub(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+) \${chars[2]}([^$]+)( ?)", r"(\1-\2-\3)\4", field)
        field = liste_subfields(field)

        i = 0
        
        for subfield in field:
            try:
                rule = rules[subfield[0]]
            except KeyError:
                rule = f" -- {subfield[0]}"
            
            if ("(" in rule and "(" in subfield[1]) or (")" in rule and ")" in subfield[1]):
                rule = f" {subfield[0]}"
            # print(70, re.sub(f"\${char}([^$]+)", rule.replace(subfield[0], subfield[1]), subfield[1]))
            field[i] = rule.replace(subfield[0], subfield[1])
            
            i += 1
        field = "".join(field)
        field = field.replace("( ", "(").replace(" )", ")")
        field = field.replace(")(", ' ; ')
        field = field.replace(" ;; ", ' ; ').replace(" ; ; ", ' ; ')
        field = field.replace(";)", ')')
        field = field.replace("((", '(').replace("))", ")")
        while "  " in field:
            field = field.replace("  ", " ")
        if re.search("\$[\w]", field) is not None:
            field = re.sub("\$[\w]", " ¤¤ ", field)
    field = field.strip()
    
    return field

#### Classe Auteur
Attention, ici, on récupère le NNA BnF en interrogeant directement le SRU de la BnF avec le critère aut.isni. 
Ce n'est cependant peut-être pas la meilleure solution (on pourrait aussi imaginer un tableau de correspondance validé par le SRE, sans doute plus fiable).
**Il est impératif de valider le processus final avec Iris Berbain**

In [126]:
class Auteur:
    """A partir d'une zone unimarc de type 7XX, on définit les attributs à intégrer dans les colonnes Auteur de la matrice
    On en fait un objet à part pour garder ensemble les informations liées à un même auteur, quel que soit le nombre d'auteurs qu'on voudrait au final"""
    def __init__(self, field):
        
        self.tag = field.tag if field else ""
        self.field = field if field else ""
        self.nom = next(iter(field.get_subfields("a")), "") if field else ""
        self.dol_o = next(iter(field.get_subfields("o")), "") if field else ""
        self.isni = ""
        if self.dol_o.startswith("ISNI") :
            self.isni = self.dol_o[4:]
           
        self.nna = ""
        #Récupération du NNA à partir de l'isni trouvé par l'équipe de la BML
        if self.isni != "":
            print("IN")
            # record_aut = sru.SRU_result(f"aut.isni any '{self.isni}'", parametres = {"recordSchema":"intermarcxchange"}).firstRecord
            result = sru.SRU_result(
                    f"aut.isni any '{self.isni}'",
                    parametres={"recordSchema": "intermarcxchange"}
            )
            print(result.test, result.url, result.errors)

            if result.test:
                record_aut = result.firstRecord
            else:
                record_aut = None
            if record_aut is not None:
                self.nna = sru.record2fieldvalue(record_aut, "001")[5:]
            
            
        #La matrice ne gère pas les codes fonction répétés, on ne prend donc que le premier
        
        self.fonction = next(iter(field.get_subfields("4")), "") if field else ""
    
        #Conversion des codes fonction Unimarc 
        self.fonction = CODES_FONCTION[self.fonction] if (self.fonction in CODES_FONCTION and self.fonction != "") else ""
        #si un code fonction n'est pas dans la liste prévue, il devient indéterminé par défaut
        if self.fonction == "" and self.nom != "":
            self.fonction = "9990"
       
       
        self.type = "Collectivité" if self.tag in ["710", "711", "712"] else ""        
        self.type = "Personne" if self.tag in ["700", "702", "701", "720"] else self.type
        
        
        self.prenom = "" if field else ""

        


        
        if self.type =="Collectivité":
            self.prenom = next(iter(field.get_subfields("b")), "") if field else ""
            
            
        if self.type == "Personne":
            self.prenom = next(iter(field.get_subfields("g")), "") if field else ""
            if self.prenom == "":
                self.prenom = next(iter(field.get_subfields("b")), "") if field else ""
            
   


#### Classe notice unimarc
Cette classe permet de constituer un objet avec les données à intégrer dans la matrice
Attention aux commentaires :
- les commentaires entre guillemet sont des explications
- les commentaires commençant par dièse ont vocation à être des lignes de code permettant de récupérer des données du lot final absentes de l'échantillon utilisé pour les tests

In [127]:
class notice_unimarc:
    
   
    def __init__(self, marc_record):
        self.marc_record = marc_record
        
        
       
        self.provenance = "BM de Lyon"
       
        self.num_notice = marc_record["001"].data
        self.ark = ""
        """ Attention ! Retier dièse sur la ligne suivante pour récupérer l'ark de la notice (absent de l'échantillon, mais présent dans le lot final)"""
        self.ark = marc_record["003"].data

        """Auteurs"""      
        champs_prioritaires = ["700", "710", "720", "701", "711", "721", "702", "712", "722"]
        auteurs = []
        for tag in champs_prioritaires:
            for field in marc_record.get_fields(tag):
                auteur = Auteur(field) if field else None
                if auteur and auteur not in auteurs and len(auteurs) <3:
                        auteurs.append(auteur)

        self.Auteur1 = auteurs[0] if len(auteurs)>0 else Auteur(None)
        self.Auteur2 = auteurs[1] if len(auteurs)>1 else Auteur(None)
        self.Auteur3 = auteurs[2] if len(auteurs)>2 else Auteur(None)
        
    
        
        
        """Langues"""
        self.liste_langues =  marc_record.get_fields("101")[0].get_subfields("a") if len(marc_record.get_fields("101"))>0 else []
        self.langue1 = self.liste_langues[0] if len(self.liste_langues) > 0 else None
        self.langue2 = self.liste_langues[1] if len(self.liste_langues) > 1 else None
        self.langue3 = self.liste_langues[2] if len(self.liste_langues) > 2 else None
        if self.langue1 not in CODES_LANGUE :
            self.langue1 = "und"
        if self.langue2 and self.langue2 not in CODES_LANGUE:
            self.langue2 = "und"
        if self.langue3 and self.langue3 not in CODES_LANGUE: 
            self.langue3 = "und"

       

        
        """Attributs liés à publication et à l'impression
        Vu avec BmL : l'adresse bibliographique n'est pas systématiquement dans la même sous-zone, on en teste différentes"""
        ordre_adresse = [("210", "a"), ("210", "e"), ("214", "r")]
        self.lieu_edition = ""
        for subfield in ordre_adresse:
            if self.lieu_edition == "":
                self.lieu_edition = marc_record.get_fields(subfield[0])[0].get_subfields(subfield[1])[0] if len(marc_record.get_fields(subfield[0]))>0 and len(marc_record.get_fields(subfield[0])[0].get_subfields(subfield[1]))>0 else ""

       
        self.date_edition = "[s. d.]"
        self.editeur = "[s. n.]"
        self.lieu_impression = ""
        self.date_impression = ""

        if len(marc_record.get_fields("210"))>0 :
            # Valeurs par défaut "[s. d.]" / "[s. n.]" si la sous-zone est absente
            self.date_edition = marc_record.get_fields("210")[0].get_subfields("d")[0] if len(marc_record.get_fields("210")[0].get_subfields("d")) >0 else "[s. d.]"
            self.editeur = marc_record.get_fields("210")[0].get_subfields("c")[0] if len(marc_record.get_fields("210")[0].get_subfields("c"))>0 else "[s. n.]"
            self.lieu_impression = marc_record.get_fields("210")[0].get_subfields("e")[0] if len(marc_record.get_fields("210")[0].get_subfields("e"))>0 else ""
            self.date_impression = marc_record.get_fields("210")[0].get_subfields("g")[0] if len(marc_record.get_fields("210")[0].get_subfields("g"))>0 else ""

        
        """Date normalisée """
        self.date = marc_record.get_fields("100")[0].get_subfields("a")[0][9:17]if len(marc_record.get_fields("100"))>0 and len(marc_record.get_fields("100")[0].get_subfields("a"))>0   else ""
        regex_date = r"[0-9\.]{4}([0-9\. ]{4})?" # ELLES MARCHENT VRAIMENT LEURS REGEX?
        search = re.fullmatch(regex_date, self.date)
        if search is None :
            self.date = re.sub(r"[^0-9\. ]", ".", self.date)
        search2 = re.fullmatch(r"[0-9\.]{4}(\.)\.{3}", self.date)
        if search2 is not None :
            self.date = re.sub(r"[0-9\.]{4}(\.)\.{3}", "1", self.date)
        

        
        """Code pays"""
        self.pays = ""
        if len(marc_record.get_fields("102"))>0 :
            self.pays = marc_record.get_fields("102")[0].get_subfields("a")[0] if len(marc_record.get_fields("102")[0].get_subfields("a"))>0 else ""
        
        self.pays = self.pays.lower()
        if self.pays not in CODES_PAYS:
            self.pays = "xx"
       
        """Zone de titre"""
        self.titre = marc_record.get_fields("200")[0].get_subfields("a")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("a"))>0   else ""
        self.complement_titre = marc_record.get_fields("200")[0].get_subfields("e")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("e"))>0   else ""
        self.titre_parallele = marc_record.get_fields("200")[0].get_subfields("d")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("d"))>0   else ""
        self.mention_resp_1 = marc_record.get_fields("200")[0].get_subfields("f")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("f"))>0   else ""
        # I02-02 / I02-03 : 200$g, première et deuxième occurrence
        liste_200g = marc_record.get_fields("200")[0].get_subfields("g") if len(marc_record.get_fields("200"))>0 else []
        self.mention_resp_2 = liste_200g[0] if len(liste_200g) > 0 else ""
        self.mention_resp_3 = liste_200g[1] if len(liste_200g) > 1 else ""

        """Informations techniques"""
        self.tirage = marc_record.get_fields("205")[0].get_subfields("a")[0] if len(marc_record.get_fields("205"))>0 and len(marc_record.get_fields("205")[0].get_subfields("a"))>0   else ""      
        self.importance_materielle = marc_record.get_fields("215")[0].get_subfields("a")[0] if len(marc_record.get_fields("215"))>0 and len(marc_record.get_fields("215")[0].get_subfields("a"))>0   else ""
        self.couleur = marc_record.get_fields("215")[0].get_subfields("c")[0] if len(marc_record.get_fields("215"))>0 and len(marc_record.get_fields("215")[0].get_subfields("c"))>0   else "" 
        self.dimensions = marc_record.get_fields("215")[0].get_subfields("d")[0] if len(marc_record.get_fields("215"))>0 and len(marc_record.get_fields("215")[0].get_subfields("d"))>0   else ""

        self.categorie_technique = marc_record.get_fields("200")[0].get_subfields("b")[0][1:-1] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("b"))>0   else "autre catégorie technique"
        # print(marc_record.get_fields("200")[0].get_subfields("b")[0])
        self.categorie_technique = CATEGORIE_TECHNIQUE[self.categorie_technique] if self.categorie_technique in CATEGORIE_TECHNIQUE else "autre catégorie technique"
        # print(self.categorie_technique)
        
        self.coordo_geographiques = marc_record.get_fields("206")[0].get_subfields("d")[0] if len(marc_record.get_fields("206"))>0 and len(marc_record.get_fields("206")[0].get_subfields("d"))>0   else ""     
     
                   

        
        """Indexation : on a deux systèmes différents selon que le document fait partie de Photographes en RA ou non"""
        self.indexation = []
        

        photographes_ra = False

        z461 =  marc_record.get_fields("461")
        for field in z461 :
            if  "Photographes en Rhône-Alpes" in next(iter(field.get_subfields("t")), ""):
                photographes_ra = True


        if photographes_ra :
            zones_indexation = [("608", 1), ("600", 3), ("606", 3)]
            for zone in zones_indexation: 
                    fields = marc_record.get_fields(zone[0])
                    i = 0
                    while i < zone[1] and i <len(fields):
                        
                        str_field = field_to_string(fields[i])
                        self.indexation.append(marcfield2label(str_field))
                        i+= 1
            liste_617 = fields = marc_record.get_fields("617")[:2]
            i = 0
            while len(self.indexation)<15 and i <len(liste_617):
                
                str_field = field_to_string(liste_617[i])
                self.indexation.append(marcfield2label(str_field))
                i += 1
                
            
        else : 
            # LA PARTIE MARRANTE, A BIEN VERIFIER!!!!
            zones_indexation = [("608", 1), ("600", 3), ("607", 2), ("606", 5), ("610", 1)]
            
            for zone in zones_indexation: 
                fields = marc_record.get_fields(zone[0])
                i = 0
                while i < zone[1] and i <len(fields):
                    str_field = field_to_string(fields[i])
                    self.indexation.append(marcfield2label(str_field))
                    i+= 1
    
      
            for zone in zones_indexation:
                fields = marc_record.get_fields(zone[0])
                i = 0
                while  i <len(fields) and len(self.indexation)<15: 
                    str_field = field_to_string(fields[i])
                    str_field = marcfield2label(str_field)
                    if str_field not in self.indexation :
                        self.indexation.append(str_field)                     
                    i += 1
                        
           
        self.notes = ""
        zones_notes = ["300", "305", "307", "461", "488", "980"]

        # VERIF CE TRUC!!!
        # hmmm... CA M'EMBETE CE CODE!
        for zone in zones_notes:            
            fields = marc_record.get_fields(zone)                
            for field in fields:                    
                if zone == "300" :
                    note =  next(iter(field.get_subfields("a")), "")
                    if note != "":
                        self.notes += "Note générale : "
                        self.notes  += note if note.endswith(".") else note + ". "
                elif zone == "305":
                    note =  next(iter(field.get_subfields("a")), "")
                    if note != "":
                        self.notes += "Etats : "
                        self.notes  += note if note.endswith(".") else note + ". "
                elif zone == "307":
                    note =  next(iter(field.get_subfields("a")), "") 
                    if note != "":
                        self.notes += "Note sur les techniques : "
                        self.notes  += note if note.endswith(".") else note + ". "
                elif zone == "461":
                    note =  next(iter(field.get_subfields("t")), "")
                    if note != "":
                        self.notes += "Fait partie d'une série : "
                        self.notes  += note if note.endswith(".") else note + ". " 
                elif zone == "488":
                    note =  next(iter(field.get_subfields("t")), "") + " : " + next(iter(field.get_subfields("i")), "")
                    
                    if note != " : ":
                        self.notes += "En lien avec le document : "
                        self.notes  += note if note.endswith(".") else note + ". " 
                elif zone == "980":
                    note =  next(iter(field.get_subfields("a")), "")
                    if note != "":
                        self.notes += "Note de contexte historique : "
                        self.notes  += note if note.endswith(".") else note + ". " 

     

               
        """Notes d'exemplaire"""     
        self.note_exemplaires = ""
        self.z316 = marc_record.get_fields("316")
        self.z317 = marc_record.get_fields("317")
        self.note_exemplaires = note_exe2string(self.z316, self.note_exemplaires)
        self.note_exemplaires = note_exe2string(self.z317, self.note_exemplaires)
        
        """ Cote et code-barre
       Dans le fichier test utilisé pour faire le script, il n'y a ni code ni code-barre, or cces deux informations sont 
       essentielles pour le résultat final 
       => retirer les dièses sur les deux lignes suivantes pour intégrer les cotes et codes barre du fichier définitif"""
        self.cote = ""
       
        self.cote = marc_record.get_fields("930")[0].get_subfields("a")[0] if len(marc_record.get_fields("930"))>0 and len(marc_record.get_fields("930")[0].get_subfields("a"))>0   else ""
        self.code_barre = ""
        self.code_barre = marc_record.get_fields("017")[0].get_subfields("a")[0] if len(marc_record.get_fields("017"))>0 and len(marc_record.get_fields("017")[0].get_subfields("a"))>0   else ""
        print(marc_record.get_fields("017")[0].get_subfields("a")[0])
        print(self.code_barre)
        """Codes projets 
        On initie la liste avec le code projet générique, voir ici s'il faut en ajouter d'autres ?"""
        self.codes_projets = "LyoNu000" # ["LyoNu000"]
        
    
    
            
    def get_subfield(self, field_tag, subfield_tag):
        #Récupère la valeur d'une sous-zone. Attention, ne fonctionne que si la zone et la sous-zone sont non répétables !
        record = self.marc_record
        field = record.get_fields(field_tag)[0] if record.get_fields(field_tag) else None
               
        res = ""
        if field is not None:
            
           
            res = field.get_subfields(subfield_tag)[0] if field.get_subfields(subfield_tag) else ""
        return res
        
    def __str__(self) :
        string = ""
        return string


## Création d'une liste de notices à partir du fichier ISO2709

In [128]:
list_indexation = []

In [129]:
list_records = []

In [130]:
#Changer ici le chemin du fichier avec le fichier définitif

# CORRECTIF ENCODAGE (ajouté) : les fichiers GMB sont en UTF-8, mais leur label (position 9) est vide,
# donc MARCReader(force_utf8=False) les décode en MARC-8 et produit des caractères erronés (é -> ©♭).
# Avec force_utf8=True, toutes les notices sont rejetées (None) à cause d'octets parasites non UTF-8
# (ex. \x03\xa2 en fin d'URL). On lit donc les notices une à une, on neutralise ces octets
# (remplacés par des espaces, sans changer la longueur) et on force le décodage UTF-8.
from pymarc import Record

def lire_records_utf8(fh):
    for chunk in fh.read().split(b'\x1d'):  # \x1d = fin de notice ISO 2709
        if len(chunk) < 24:
            continue
        buf = bytearray(chunk)
        while True:
            try:
                bytes(buf).decode('utf-8')
                break
            except UnicodeDecodeError as e:
                for k in range(e.start, e.end):
                    buf[k] = 0x20
        yield Record(data=bytes(buf) + b'\x1d', to_unicode=True, force_utf8=True)

# AJOUT : nom du fichier à convertir, réutilisé pour nommer le fichier Excel final
# (ex. 'GMB_Fichier2' -> GMB_Fichier2_partie_1.xlsx). Changer uniquement ici.
NOM_FICHIER = 'GMB_Fichier6'

# idx=0

with open(NOM_FICHIER, 'rb') as fh:  # MODIF : 'GMB_Fichier2' remplacé par NOM_FICHIER
    reader = lire_records_utf8(fh)  # CORRECTIF : remplace MARCReader(fh, force_utf8=False)
    for record in tqdm(reader,desc="Processing Records"):
        notice = notice_unimarc(record)

        list_records.append(notice)

        list_indexation.extend(notice.indexation)   
        # idx += 1
        # if idx > 150: break
         

Processing Records: 172it [00:00, 883.34it/s]

BML:OTC_05UIO000101V42049
BML:OTC_05UIO000101V42049
BML:OTC_05UIO000101V22559
BML:OTC_05UIO000101V22559
BML:OTC_05UIO000101V22532
BML:OTC_05UIO000101V22532
BML:OTC_05UIO000101V22535
BML:OTC_05UIO000101V22535
BML:OTC_05UIO000101V22533
BML:OTC_05UIO000101V22533
BML:OTC_05UIO000101V36739
BML:OTC_05UIO000101V36739
BML:OTC_05UIO000101V36740
BML:OTC_05UIO000101V36740
BML:OTC_05UIO000101V36742
BML:OTC_05UIO000101V36742
BML:OTC_05UIO000101V36741
BML:OTC_05UIO000101V36741
BML:OTC_05UIO000101V36728
BML:OTC_05UIO000101V36728
BML:OTC_05UIO000101V36727
BML:OTC_05UIO000101V36727
BML:OTC_05UIO000101V36729
BML:OTC_05UIO000101V36729
BML:OTC_05UIO000101V36724
BML:OTC_05UIO000101V36724
BML:OTC_05UIO000101V36723
BML:OTC_05UIO000101V36723
BML:OTC_05UIO000101V36726
BML:OTC_05UIO000101V36726
BML:OTC_05UIO000101V36725
BML:OTC_05UIO000101V36725
BML:OTC_05UIO000101V36722
BML:OTC_05UIO000101V36722
BML:OTC_05UIO000101V21642
BML:OTC_05UIO000101V21642
BML:OTC_05UIO000101V23238
BML:OTC_05UIO000101V23238
BML:OTC_05UI

Processing Records: 437it [00:00, 832.24it/s]

BML:OTC_05UIO000101V20854
BML:OTC_05UIO000101V20854
BML:OTC_05UIO000101V20853
BML:OTC_05UIO000101V20853
BML:OTC_05UIO000101V20852
BML:OTC_05UIO000101V20852
BML:OTC_05UIO000101V20851
BML:OTC_05UIO000101V20851
BML:OTC_05UIO000101V20850
BML:OTC_05UIO000101V20850
BML:OTC_05UIO000101V20849
BML:OTC_05UIO000101V20849
BML:OTC_05UIO000101V20848
BML:OTC_05UIO000101V20848
BML:OTC_05UIO000101V20857
BML:OTC_05UIO000101V20857
BML:OTC_05UIO000101V20856
BML:OTC_05UIO000101V20856
BML:OTC_05UIO000101V20855
BML:OTC_05UIO000101V20855
BML:OTC_05UIO000101V20871
BML:OTC_05UIO000101V20871
BML:OTC_05UIO000101V20870
BML:OTC_05UIO000101V20870
BML:OTC_05UIO000101V20869
BML:OTC_05UIO000101V20869
BML:OTC_05UIO000101V36757
BML:OTC_05UIO000101V36757
BML:OTC_05UIO000101V23743
BML:OTC_05UIO000101V23743
BML:OTC_05UIO000101V23744
BML:OTC_05UIO000101V23744
BML:OTC_05UIO000101V23730
BML:OTC_05UIO000101V23730
BML:OTC_05UIO000101V23768
BML:OTC_05UIO000101V23768
BML:OTC_05UIO000101V23770
BML:OTC_05UIO000101V23770
BML:OTC_05UI

Processing Records: 724it [00:00, 911.94it/s]

BML:OTC_05UIO000101V23847
BML:OTC_05UIO000101V23847
BML:OTC_05UIO000101V23848
BML:OTC_05UIO000101V23848
BML:OTC_05UIO000101V23893
BML:OTC_05UIO000101V23893
BML:OTC_05UIO000101V23884
BML:OTC_05UIO000101V23884
BML:OTC_05UIO000101V23890
BML:OTC_05UIO000101V23890
BML:OTC_05UIO000101V23880
BML:OTC_05UIO000101V23880
BML:OTC_05UIO000101V23881
BML:OTC_05UIO000101V23881
BML:OTC_05UIO000101V23882
BML:OTC_05UIO000101V23882
BML:OTC_05UIO000101V23883
BML:OTC_05UIO000101V23883
BML:OTC_05UIO000101V23871
BML:OTC_05UIO000101V23871
BML:OTC_05UIO000101V23873
BML:OTC_05UIO000101V23873
BML:OTC_05UIO000101V23876
BML:OTC_05UIO000101V23876
BML:OTC_05UIO000101V23877
BML:OTC_05UIO000101V23877
BML:OTC_05UIO000101V23878
BML:OTC_05UIO000101V23878
BML:OTC_05UIO000101V25110
BML:OTC_05UIO000101V25110
BML:OTC_05UIO000101V23372
BML:OTC_05UIO000101V23372
BML:OTC_05UIO000101V23375
BML:OTC_05UIO000101V23375
BML:OTC_05UIO000101V23376
BML:OTC_05UIO000101V23376
BML:OTC_05UIO000101V23379
BML:OTC_05UIO000101V23379
BML:OTC_05UI

... (88 outputs left out)

Processing Records: 11385it [00:14, 530.55it/s]

BML:OTC_05UIO000101V25517
BML:OTC_05UIO000101V25517
BML:OTC_05UIO000101V25524
BML:OTC_05UIO000101V25524
BML:OTC_05UIO000101V25523
BML:OTC_05UIO000101V25523
BML:OTC_05UIO000101V25526
BML:OTC_05UIO000101V25526
BML:OTC_05UIO000101V25525
BML:OTC_05UIO000101V25525
BML:OTC_05UIO000101V18633
BML:OTC_05UIO000101V18633
BML:OTC_05UIO000101V18631
BML:OTC_05UIO000101V18631
BML:OTC_05UIO000101V18632
BML:OTC_05UIO000101V18632
BML:OTC_05UIO000101V42453
BML:OTC_05UIO000101V42453
BML:OTC_05UIO000101V42454
BML:OTC_05UIO000101V42454
BML:OTC_05UIO000101V42455
BML:OTC_05UIO000101V42455
BML:OTC_05UIO000101V42449
BML:OTC_05UIO000101V42449
BML:OTC_05UIO000101V42450
BML:OTC_05UIO000101V42450
BML:OTC_05UIO000101V42480
BML:OTC_05UIO000101V42480
BML:OTC_05UIO000101V42481
BML:OTC_05UIO000101V42481
BML:OTC_05UIO000101V42426
BML:OTC_05UIO000101V42426
BML:OTC_05UIO000101V42425
BML:OTC_05UIO000101V42425
BML:OTC_05UIO000101V25641
BML:OTC_05UIO000101V25641
BML:OTC_05UIO000101V42190
BML:OTC_05UIO000101V42190
BML:OTC_05UI

Processing Records: 11562it [00:14, 790.78it/s]

BML:OTC_05UIO000101V21304
BML:OTC_05UIO000101V21304
BML:OTC_05UIO000101V21305
BML:OTC_05UIO000101V21305
BML:OTC_05UIO000101V42038
BML:OTC_05UIO000101V42038
BML:OTC_05UIO000101V42042
BML:OTC_05UIO000101V42042
BML:OTC_05UIO000101V42041
BML:OTC_05UIO000101V42041
BML:OTC_05UIO000101V42040
BML:OTC_05UIO000101V42040
BML:OTC_05UIO000101V42039
BML:OTC_05UIO000101V42039
BML:OTC_05UIO000101V42037
BML:OTC_05UIO000101V42037
BML:OTC_05UIO000101V42036
BML:OTC_05UIO000101V42036
BML:OTC_05UIO000101V42225
BML:OTC_05UIO000101V42225
BML:OTC_05UIO000101V42226
BML:OTC_05UIO000101V42226
BML:OTC_05UIO000101V42230
BML:OTC_05UIO000101V42230
BML:OTC_05UIO000101V42229
BML:OTC_05UIO000101V42229
BML:OTC_05UIO000101V42228
BML:OTC_05UIO000101V42228
BML:OTC_05UIO000101V42227
BML:OTC_05UIO000101V42227
BML:OTC_05UIO000101V42233
BML:OTC_05UIO000101V42233
BML:OTC_05UIO000101V42232
BML:OTC_05UIO000101V42232
BML:OTC_05UIO000101V42231
BML:OTC_05UIO000101V42231
BML:OTC_05UIO000101V42234
BML:OTC_05UIO000101V42234
BML:OTC_05UI

In [131]:
print(len(list_records))

11562


In [132]:
counter = Counter(list_indexation)
subjects = [(count, subject) for subject, count in counter.items()]
df_subjects = pandas.DataFrame(subjects, columns = ["Count", "Subjects"])
df_subjects.to_excel("./test_indexation.xlsx")

## Création de la matrice

### Mapping
Concordance des informations entre les colonnes de la matrice et les attributs de l'objet notice unimarc

In [133]:
record2matrice = {
    "I01-01": "titre",
    "I01-02" : "complement_titre",
    "I01-03" : "titre_parallele",
    "I02-01" : "mention_resp_1",
    "I02-02" : "mention_resp_2",
    "I02-03" : "mention_resp_3",
    "I12-01" : "langue1", 
    "I12-02" : "langue2", 
    "I12-03" : "langue3",
    "I21-01" : "Auteur1.nna",
    "I21-02" : "Auteur1.nom",
    "I21-03" : "Auteur1.prenom",
    "I21-04" : "Auteur1.fonction",
    "I21-05" : "Auteur1.type",
    "I22-01" : "Auteur2.nna",
    "I22-02" : "Auteur2.nom",
    "I22-03" : "Auteur2.prenom",
    "I22-04": "Auteur2.fonction",
    "I22-05": "Auteur2.type",
    "I23-01" : "Auteur3.nna",
    "I23-02" : "Auteur3.nom",
    "I23-03" : "Auteur3.prenom",
    "I23-04": "Auteur3.fonction",
    "I23-05": "Auteur3.type",
    "I32-01" : "date_edition", # ????????
    "I31-02" : "lieu_edition",
    "I31-03" : "editeur",
    "I33-01" : "lieu_impression",
    "I33-03" : "date_impression",
    "I32-03" : "date",
    "I41-01" : "tirage",
    "I42-01": "importance_materielle",
    "I42-03" : "couleur", 
    "I42-04" : "dimensions",
    "I43-03" : "categorie_technique",
    "I46-02" : "coordo_geographiques",
    "I51-01" : "notes",
    "I11-01" : "pays",
    "I71-01": "codes_projets",
    # "I71-02" : "", # A DEFINIR AVEC BML 
    # "I71-03": "", # A DEFINIR AVEC BML
    # le I71-01 est géré de manière différencié car ne varie jamais
    "I72-03" : "cote",
    "I72-04" : "ark",
    "I72-05" : "code_barre",
    "I73-01" : "note_exemplaires",
}

len(record2matrice)


43

### Liste des colonnes

In [ ]:
COLS_MATRICE = ["I01-01", "I01-02", "I01-03", "I01-04", "I02-01", 
                "I02-02", "I02-03", "I11-01", "I12-01", "I12-02", 
                "I12-03", "I21-01", "I21-02", "I21-03", "I21-04", 
                "I21-05", "I22-01", "I22-02", "I22-03", "I22-04", 
                "I22-05", "I23-01", "I23-02", "I23-03", "I23-04", 
                "I23-05", "I31-02", "I31-03", "I32-01", "I32-03", 
                "I33-01", "I33-02", "I33-03", "I41-01", "I42-01", 
                "I42-03", "I42-04", "I43-01", "I43-02", "I43-03", 
                "I44-01", "I45-01", "I51-01", "I52-01", "I71-01", 
                "I71-02", "I71-03", "I72-02", "I72-03", "I72-04", 
                "I72-05", "I73-01", "I73-02", "I73-03"]

len(COLS_MATRICE)

54

### Transformation des notices en matrice

In [135]:
def col2attr(key, record, dic) : 
    """Fonction qui permet d'aller récupérer les attributs à partir du dictionnaire ci-dessous"""
    if "." in dic[key] :
        el = dic[key].split(".")
        auteur = getattr(record, el[0])
        res = getattr(auteur, el[1])
    else : 
        res = getattr(record, dic[key])
    return res

In [136]:
i = 1
records_data = []
for record in tqdm(list_records,desc="Processing Records") :
    d = {}
    
    d["id"] = record.num_notice
   
    d["I72-01"] = record.provenance
  
    
    for col in COLS_MATRICE:
        if col in record2matrice :
            d[col] = col2attr(col, record, record2matrice)
        else :
            d[col] = ""
    for i, item in enumerate(record.indexation):
        if i<10:
            d[f"I61-00-0{i}"] = item
        else :
            d[f"I61-00-{i}"] = item
    # PB : le code ci-dessous créé un champ I71-00 qui en théorie devrait plutôt être I71-01; 
    # for i, item in enumerate(record.codes_projets):
    #     if i<10:
    #         d[f"I71-0{i}"] = item
    #     else :
    #         d[f"I71-{i}"] = item
   
       
    
   
  
    records_data.append(d)
    #matrice = pandas.concat([matrice, pandas.DataFrame([d])], ignore_index=True)
    #print(i, "/", len(list_records))
    i+= 1

matrice = pandas.DataFrame(records_data)
    
        

Processing Records: 100%|██████████| 11562/11562 [00:00<00:00, 17649.85it/s]


In [137]:
#Pour avoir les colonnes dans le bon ordre 
df = matrice.sort_index(axis=1)

### Export : à utiliser si petit fichier test

In [138]:
# df.to_excel("./matrice_lot_2.xlsx")

### Export : à utiliser pour + de 50 000 lignes

In [139]:
def clean_excel_illegal_chars(df):
    return df.map(
        lambda x: ILLEGAL_CHARACTERS_RE.sub("", str(x)) if isinstance(x, str) else x
    )

In [140]:
chunk_size = 50000

# Découpage en fichiers Excel
for i in range(0, len(df), chunk_size):
    chunk = df.iloc[i:i + chunk_size]
    chunk = clean_excel_illegal_chars(chunk) 
    chunk.to_excel(f"./{NOM_FICHIER}_partie_{i//chunk_size + 1}.xlsx", index=False)  # MODIF : nom du fichier GMB dans le nom de sortie



In [141]:
import random

def comparer_echantillon_aleatoire(list_records, df, n=5):
    """
    Prend un échantillon aléatoire de n notices et compare linéairement 
    la notice UNIMARC brute avec les colonnes remplies dans le DataFrame Excel.
    """
    sample_size = min(n, len(list_records))
    echantillon = random.sample(list(enumerate(list_records)), sample_size)
    
    for idx, notice in echantillon:
        print("=" * 95)
        print(f"NOTICE UNIMARC N° {notice.num_notice} (Index de la liste : {idx})")
        print("=" * 95)
        
        print("\n[CHAMPS UNIMARC ORIGINAUX]")
        idx_field=0
        for field in notice.marc_record.fields:
            print(f"  {field}")
            idx_field+=1
            
        print("\n[COLONNES EXCEL ASSOCIÉES (Valeurs non vides)]")
        idx_excel=0
        row = df[df["id"] == notice.num_notice]
        if not row.empty:
            for col in df.columns:
                val = row.iloc[0][col]
                if val != "" and pandas.notna(val):
                    print(f"  {col} : {val}")
                    idx_excel += 1
        else:
            print("  [Avertissement] ID introuvable dans le DataFrame Excel.")
        print("\n")

    print(f"idx_field: {idx_field}, idx_excel: {idx_excel}")

# Exécution de la comparaison sur un échantillon de 5 éléments
comparer_echantillon_aleatoire(list_records, df, n=1)

NOTICE UNIMARC N° 0003348289 (Index de la liste : 10318)

[CHAMPS UNIMARC ORIGINAUX]
  =001  0003348289
  =003  https://catalogue.bm-lyon.fr/ark:/75584/pf0003348289.locale=fr\
  =005  20261005213429.0
  =017  70$aBML:OTC_05UIO000101V26259$2PID Fedora
  =100  \\$a20260922d1988    k  y0frey01      ba
  =116  \\$afiybxx    xx    zz
  =200  10$aVillage d’Arindj$b[Photographie]
  =210  \\$as. l.$cs. n.$d[1988]
  =215  \\$a1 négatif noir et blanc$d2,4 x 3,6 cm
  =300  \\$aStèle à croix du cimetière
  =317  \\$aDépôt de l’Union Internationale des Organisations Terre et Culture (UIOTC). Fonds Images du patrimoine arménien. Fonds Argam Ayvazian$5FR693836101 : V-26259
  =319  \\$aToute reproduction est interdite sans l'autorisation de l'UIOTC$5FR693836101 : NUM_V-26259
  =461  \1$tSérie République d'Arménie
  =608  \\$aReprésentations d’objet$2frTICO
  =610  0\$astèle à croix 
  =610  0\$akhatchkar
  =617  \\$bAyrarat$cKotayk’$dAṙinǰ
  =617  \\$bԱյրարատ$cԿոտայք$dԱռինջ
  =617  \\$cKotayk‘$dAṙind

In [142]:
NOTICES = list(enumerate(list_records))
notice = NOTICES[0][1]

for field in notice.marc_record.fields:
    print(f"  {field}")

row = df[df["id"] == notice.num_notice]
if not row.empty:
    for col in df.columns:
        val = row.iloc[0][col]
        if val != "" and pandas.notna(val):
            print(f"  {col} : {val}")
            

  =001  0003350241
  =003  https://catalogue.bm-lyon.fr/ark:/75584/pf0003350241.locale=fr\
  =005  20261005213356.0
  =017  70$aBML:OTC_05UIO000101V42049$2PID Fedora
  =100  \\$a20260922d19901999k  y0frey01      ba
  =116  \\$afiybxx    xx    zz
  =200  10$aVille de Vagharchapat$b[Photographie]
  =210  \\$as. l.$cs. n.$d[1990-1999]
  =215  \\$a1 photographie positive $ctirage noir et blanc
  =300  \\$aCathédrale d'Etchmiadzin
  =300  \\$aAncienne cote : 9718
  =317  \\$aDépôt de l’Union Internationale des Organisations Terre et Culture (UIOTC). Fonds Images du patrimoine arménien. Fonds Argam Ayvazian$5FR693836101 : V-42049
  =319  \\$aToute reproduction est interdite sans l'autorisation de l'UIOTC$5FR693836101 : NUM_V-42049
  =461  \1$tSérie République d'Arménie
  =608  \\$aVues d’architecture$2frTICO
  =610  0\$acathédrale
  =617  \\$bAyrarad$cAragaçotn$dVaġaršapat
  =617  \\$bԱյրարատ$cԱրագածոտն$dՎաղարշապատ
  =617  \\$cAragatzotn$dVagharchapat
  =617  \\$cArakadzodn$dVagharchabad
  